In [1]:
pip install numpy sentence-transformers faiss-cpu pypdf python-docx

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import numpy as np
import faiss

from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer


# ============================================================
# 1. LOAD SENTENCE TRANSFORMER MODEL
# ============================================================

print("Loading Sentence Transformer model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")


# ============================================================
# 2. EXTRACT TEXT FROM PDF / DOCX / TXT
# ============================================================

def extract_text(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    # ---------------- PDF ----------------
    if extension == ".pdf":

        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        return text

    # ---------------- DOCX ----------------
    elif extension == ".docx":

        document = Document(file_path)

        text = ""

        for paragraph in document.paragraphs:

            if paragraph.text.strip():
                text += paragraph.text + "\n"

        return text

    # ---------------- TXT ----------------
    elif extension == ".txt":

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as file:

            return file.read()

    else:

        raise ValueError(
            "Only PDF, DOCX and TXT files are supported."
        )


# ============================================================
# 3. CREATE TEXT CHUNKS
# ============================================================

def create_chunks(
    text,
    chunk_size=500,
    overlap=100
):

    # Remove unnecessary spaces
    text = " ".join(text.split())

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():

            chunks.append(chunk)

        # Move forward while keeping overlap
        start = end - overlap

    return chunks


# ============================================================
# 4. CREATE EMBEDDINGS
# ============================================================

def create_embeddings(chunks):

    embeddings = model.encode(
        chunks,
        convert_to_numpy=True
    )

    # Convert to NumPy float32
    embeddings = np.array(
        embeddings,
        dtype=np.float32
    )

    return embeddings


# ============================================================
# 5. NORMALIZE EMBEDDINGS
# ============================================================

def normalize_embeddings(embeddings):

    # Calculate vector magnitude
    norms = np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True
    )

    # Avoid division by zero
    norms[norms == 0] = 1

    # Normalize vectors
    normalized_embeddings = (
        embeddings / norms
    )

    return normalized_embeddings


# ============================================================
# 6. CREATE FAISS INDEX
# ============================================================

def create_faiss_index(embeddings):

    # Number of dimensions
    dimension = embeddings.shape[1]

    print(
        "\nEmbedding dimension:",
        dimension
    )

    # IndexFlatIP = Inner Product
    #
    # Since vectors are normalized,
    # Inner Product = Cosine Similarity

    index = faiss.IndexFlatIP(
        dimension
    )

    # Add vectors to FAISS
    index.add(embeddings)

    print(
        "FAISS index created successfully."
    )

    print(
        "Vectors stored:",
        index.ntotal
    )

    return index


# ============================================================
# 7. SPLIT TEXT INTO SENTENCES
# ============================================================

def split_sentences(text):

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    return sentences


# ============================================================
# 8. FIND BEST SENTENCES
# ============================================================

def generate_answer(
    question,
    index,
    chunks
):

    # --------------------------------------------------------
    # Convert question into embedding
    # --------------------------------------------------------

    question_embedding = model.encode(
        [question],
        convert_to_numpy=True
    )

    question_embedding = np.array(
        question_embedding,
        dtype=np.float32
    )

    # Normalize question embedding
    question_embedding = normalize_embeddings(
        question_embedding
    )

    # --------------------------------------------------------
    # Search FAISS
    # --------------------------------------------------------

    number_of_results = min(
        3,
        len(chunks)
    )

    similarities, indices = index.search(
        question_embedding,
        number_of_results
    )

    # Store candidate sentences
    candidate_sentences = []

    # --------------------------------------------------------
    # Process retrieved chunks
    # --------------------------------------------------------

    for i in range(number_of_results):

        chunk_index = indices[0][i]

        if chunk_index == -1:
            continue

        chunk = chunks[chunk_index]

        # Similarity of question to chunk
        chunk_score = float(
            similarities[0][i]
        )

        # Split chunk into sentences
        sentences = split_sentences(
            chunk
        )

        if not sentences:
            continue

        # ----------------------------------------------------
        # Create embeddings for sentences
        # ----------------------------------------------------

        sentence_embeddings = model.encode(
            sentences,
            convert_to_numpy=True
        )

        sentence_embeddings = np.array(
            sentence_embeddings,
            dtype=np.float32
        )

        # Normalize sentence embeddings
        sentence_embeddings = normalize_embeddings(
            sentence_embeddings
        )

        # ----------------------------------------------------
        # Calculate cosine similarity
        # ----------------------------------------------------

        sentence_scores = np.dot(
            sentence_embeddings,
            question_embedding[0]
        )

        # ----------------------------------------------------
        # Store sentence + score
        # ----------------------------------------------------

        for sentence, sentence_score in zip(
            sentences,
            sentence_scores
        ):

            # Combine:
            #
            # 30% chunk similarity
            # 70% sentence similarity

            final_score = (
                0.3 * chunk_score
                +
                0.7 * float(sentence_score)
            )

            candidate_sentences.append(
                (
                    sentence,
                    final_score
                )
            )

    # ========================================================
    # IF NOTHING FOUND
    # ========================================================

    if not candidate_sentences:

        return (
            "I could not find relevant information "
            "in the document.",
            0.0
        )

    # ========================================================
    # SORT BY RELEVANCE
    # ========================================================

    candidate_sentences.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # ========================================================
    # SELECT BEST SENTENCES
    # ========================================================

    selected_sentences = []
    selected_scores = []

    for sentence, score in candidate_sentences:

        if sentence not in selected_sentences:

            selected_sentences.append(
                sentence
            )

            selected_scores.append(
                score
            )

        # Maximum 3 sentences
        if len(selected_sentences) == 3:

            break

    # ========================================================
    # CREATE ONE CLEAN ANSWER
    # ========================================================

    answer = " ".join(
        selected_sentences
    )

    # Best similarity score
    relevance_score = max(
        selected_scores
    )

    return (
        answer,
        relevance_score
    )


# ============================================================
# 9. MAIN PROGRAM
# ============================================================

def main():

    print("\n============================================")
    print("       DOCUMENT QUESTION ANSWERING")
    print("============================================")

    # --------------------------------------------------------
    # GET DOCUMENT PATH
    # --------------------------------------------------------

    file_path = input(
        "\nEnter document path: "
    ).strip()

    # --------------------------------------------------------
    # CHECK FILE
    # --------------------------------------------------------

    if not os.path.exists(file_path):

        print(
            "\nERROR: File not found."
        )

        return

    # --------------------------------------------------------
    # EXTRACT TEXT
    # --------------------------------------------------------

    print(
        "\nExtracting text..."
    )

    text = extract_text(
        file_path
    )

    if not text.strip():

        print(
            "ERROR: No text found in document."
        )

        return

    print(
        "Text extracted successfully!"
    )

    print(
        "Characters extracted:",
        len(text)
    )

    # --------------------------------------------------------
    # CREATE CHUNKS
    # --------------------------------------------------------

    print(
        "\nCreating text chunks..."
    )

    chunks = create_chunks(
        text,
        chunk_size=500,
        overlap=100
    )

    print(
        "Number of chunks:",
        len(chunks)
    )

    # --------------------------------------------------------
    # CREATE EMBEDDINGS
    # --------------------------------------------------------

    print(
        "\nCreating embeddings..."
    )

    embeddings = create_embeddings(
        chunks
    )

    print(
        "Embedding shape:",
        embeddings.shape
    )

    # Example:
    #
    # (20, 384)
    #
    # 20 = number of chunks
    # 384 = embedding dimensions

    # --------------------------------------------------------
    # NORMALIZE
    # --------------------------------------------------------

    print(
        "\nNormalizing embeddings..."
    )

    embeddings = normalize_embeddings(
        embeddings
    )

    # --------------------------------------------------------
    # CREATE FAISS INDEX
    # --------------------------------------------------------

    index = create_faiss_index(
        embeddings
    )

    # ========================================================
    # READY FOR QUESTIONS
    # ========================================================

    print("\n============================================")
    print("Document is ready!")
    print("Ask questions about your document.")
    print("Type 'exit' to stop.")
    print("============================================")

    # ========================================================
    # QUESTION LOOP
    # ========================================================

    while True:

        question = input(
            "\nYour question: "
        ).strip()

        # ----------------------------------------------------
        # EXIT
        # ----------------------------------------------------

        if question.lower() == "exit":

            print(
                "\nProgram ended."
            )

            break

        # ----------------------------------------------------
        # EMPTY QUESTION
        # ----------------------------------------------------

        if not question:

            print(
                "Please enter a question."
            )

            continue

        # ----------------------------------------------------
        # GENERATE ANSWER
        # ----------------------------------------------------

        answer, relevance_score = generate_answer(
            question,
            index,
            chunks
        )

        # ====================================================
        # DISPLAY FINAL ANSWER
        # ====================================================

        print(
            "\n--------------------------------------------"
        )

        print(
            "ANSWER"
        )

        print(
            "--------------------------------------------"
        )

        print(
            answer
        )

        # ====================================================
        # DISPLAY RELEVANCE
        # ====================================================

        print(
            f"\nRelevance Score: "
            f"{relevance_score:.4f}"
        )

        print(
            f"Similarity: "
            f"{relevance_score * 100:.2f}%"
        )

        print(
            "--------------------------------------------"
        )


# ============================================================
# 10. RUN PROGRAM
# ============================================================

if __name__ == "__main__":

    main()


Loading Sentence Transformer model...
Model loaded successfully!

       DOCUMENT QUESTION ANSWERING

Enter document path: ml.txt

Extracting text...
Text extracted successfully!
Characters extracted: 655

Creating text chunks...
Number of chunks: 2

Creating embeddings...
Embedding shape: (2, 384)

Normalizing embeddings...

Embedding dimension: 384
FAISS index created successfully.
Vectors stored: 2

Document is ready!
Ask questions about your document.
Type 'exit' to stop.

Your question: What are the types of ML?

--------------------------------------------
ANSWER
--------------------------------------------
There are three major types of machine learning: supervised learning, unsupervised learning and reinforcement learning. Supervised learning uses labeled data to train a model. Machine learning is a branch of artificial intelligence.

Relevance Score: 0.4193
Similarity: 41.93%
--------------------------------------------

Your question: What are the types of Machine learning?
